# MLP Project Milestone 1 – EDA
Workspace Rental Price Tier Prediction. Run cells top to bottom; each cell matches the "Your code" box of that question.

## Q2

In [ ]:
import pandas as pd, numpy as np, json, glob
import matplotlib.pyplot as plt, seaborn as sns
def find(name):  # works on Kaggle (/kaggle/input/...) and locally
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else name
tr = pd.read_csv(find('train.csv')); te = pd.read_csv(find('test.csv'))
ss = pd.read_csv(find('sample_submission.csv'))
print(tr.shape, te.shape, ss.shape)
print(set(tr.columns) - set(te.columns), set(te.columns) - set(tr.columns))
print(tr.dtypes.value_counts()); print(tr.dtypes)
print(tr.space_id.min(), tr.space_id.max(), te.space_id.min(), te.space_id.max())
for c in tr.select_dtypes(exclude='number').columns:
    print(c, tr[c].nunique())
print('facility codes:', tr.facilities.apply(json.loads).explode().nunique())

## Q3

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
vc = tr.price_tier.value_counts().sort_index()
print(vc, (vc / len(tr)).round(4), 'max/min ratio:', round(vc.max() / vc.min(), 2))
ax = vc.plot.bar(); ax.set_title('Target distribution (train)'); plt.show()
print(pd.crosstab(tr.metro, tr.price_tier, normalize='index').round(3))
y = tr.price_tier
skf = StratifiedKFold(5, shuffle=True, random_state=0)
for s in ['most_frequent', 'stratified']:
    p = cross_val_predict(DummyClassifier(strategy=s, random_state=0), tr[['max_seats']], y, cv=skf)
    print(s, 'macro F1 =', round(f1_score(y, p, average='macro'), 4))

## Q4

In [ ]:
m = pd.DataFrame({'train': tr.isna().mean()*100, 'test': te.isna().mean()*100}).round(2)
print(m[m.train > 0].sort_values('train', ascending=False))
m[m.train > 0.1].sort_values('train').plot.barh(figsize=(6, 4)); plt.xlabel('% missing'); plt.show()
print('rows with any NA:', round(tr.isna().any(axis=1).mean(), 4))
print('empty strings:', (tr.select_dtypes(exclude='number') == '').sum().sum())
print(tr.groupby('metro').borough.apply(lambda s: s.isna().mean()))
print('scores NA <=> review_count==0:', (tr.score_overall.isna() == (tr.review_count == 0)).mean())
for c in ['borough', 'score_overall', 'zones', 'workstations', 'utility_rooms', 'operator_portfolio_size']:
    print(c, tr.groupby(tr[c].isna()).price_tier.mean().round(3).to_dict())
    print(pd.crosstab(tr[c].isna(), tr.price_tier, normalize='index').round(3))
print('max_term_days == 1125:', (tr.max_term_days == 1125).sum())

## Q5

In [ ]:
f = [c for c in tr.columns if c not in ('space_id', 'price_tier')]
print('dup ids:', tr.space_id.duplicated().sum(), 'dup rows:', tr.duplicated(f).sum())
dup = tr[tr.duplicated(f, keep=False)]
print('conflicting groups:', (dup.groupby(f, dropna=False).price_tier.nunique() > 1).sum())
print('train-test overlap:', pd.merge(tr[f].drop_duplicates(), te[f].drop_duplicates(), on=f).shape[0])
d8 = lambda c: pd.to_datetime(tr[c])
checks = {
 'min>max term': tr.min_term_days > tr.max_term_days,
 '12m>total': tr.review_count_12m > tr.review_count,
 '30d>12m': tr.review_count_30d > tr.review_count_12m,
 'first>last review': d8('first_review_date') > d8('last_review_date'),
 'last review>snapshot': d8('last_review_date') > d8('snapshot_end_date'),
 'a>b slots': tr.open_slots_a > tr.open_slots_b, 'b>c': tr.open_slots_b > tr.open_slots_c,
 'c>d': tr.open_slots_c > tr.open_slots_d,
 'max term >= 1e6': tr.max_term_days >= 1e6, 'max term == 1125': tr.max_term_days == 1125,
 'zones == 0': tr.zones == 0, 'utility 0.5': tr.utility_rooms == 0.5, 'utility > 10': tr.utility_rooms > 10,
 'metro>portfolio': tr.operator_metro_units > tr.operator_portfolio_size,
 'metro sum': tr.operator_metro_units != tr[['operator_metro_whole', 'operator_metro_private',
                                             'operator_metro_shared']].sum(axis=1),
 'active>platform age': (tr.operator_active_years*12 + tr.operator_active_months) >
                        (tr.operator_platform_age_years*12 + tr.operator_platform_age_months),
 '|lat| > 90': tr.geo_lat.abs() > 90, 'score == 10': tr.score_overall == 10}
for k, v in checks.items(): print(k, v.sum())
print(tr.score_overall.min(), tr.score_overall.max())
print((tr.min_term_days == tr.min_term_floor).mean(), (tr.max_term_days == tr.max_term_ceiling).mean())
print('snapshot gap (days):', (d8('snapshot_end_date') - d8('snapshot_start_date')).dt.days.value_counts().head())
oc = tr.operator_ref.value_counts()
print('listings/operator median:', oc.median(), 'unseen test operators:', (~te.operator_ref.dropna().isin(tr.operator_ref)).mean())

## Q6

In [ ]:
num = tr.select_dtypes('number').drop(columns=['space_id', 'price_tier'])
desc = num.describe(percentiles=[.01, .5, .99]).T
desc['skew'] = num.skew(); desc['zero%'] = (num == 0).mean() * 100
print(desc.round(2))
print(tr.max_term_days.value_counts(normalize=True).head(8))
print('share of reviewed listings with score 10:', (tr.score_overall == 10).sum() / tr.score_overall.notna().sum())
print('distinct latitudes:', tr.geo_lat.round(4).nunique())
cols = ['max_seats', 'workstations', 'operator_portfolio_size', 'review_count', 'max_term_days',
        'min_term_days', 'open_slots_d', 'booked_load', 'score_overall']
logc = {'operator_portfolio_size', 'review_count', 'max_term_days', 'min_term_days'}
fig, axs = plt.subplots(3, 3, figsize=(10, 7))
for a, c in zip(axs.flat, cols):
    s = tr[c].dropna()
    a.hist(np.log1p(s) if c in logc else s.clip(upper=s.quantile(.999)), bins=50)
    a.set_title(f'log1p({c})' if c in logc else c)
plt.tight_layout(); plt.show()

## Q7

In [ ]:
def add_features(df):  # derived columns reused in Q8-Q10
    d = df.copy()
    d['n_facilities'] = d.facilities.fillna('[]').apply(lambda s: len(json.loads(s)))
    for c in ['operator_certified', 'operator_has_avatar', 'operator_id_verified', 'bookable']:
        d[c] = d[c].map({'t': 1, 'f': 0})
    snap = pd.to_datetime(d.snapshot_start_date)
    d['days_since_last_review'] = (snap - pd.to_datetime(d.last_review_date)).dt.days
    d['listing_age_days'] = (snap - pd.to_datetime(d.first_review_date)).dt.days
    d['operator_tenure_m'] = d.operator_platform_age_years * 12 + d.operator_platform_age_months
    d['seats_per_ws'] = d.max_seats / d.workstations
    return d
d = add_features(tr)
for c in ['operator_class', 'operator_certified', 'operator_has_avatar', 'operator_id_verified',
          'access_type', 'bookable', 'compliance_status', 'metro', 'space_class']:
    print(c, tr[c].nunique(), tr[c].value_counts(normalize=True, dropna=False).head(6).round(3).to_dict())
sc = tr.space_class.value_counts()
print('top-8 space_class share:', sc.head(8).sum() / len(tr), 'classes < 30 rows:', (sc < 30).sum())
for c in ['operator_ref', 'zone', 'borough', 'space_class', 'facilities']:
    print(c, tr[c].nunique(), 'unseen in test:', round((~te[c].dropna().isin(tr[c])).mean(), 4))
fac = d.facilities.apply(json.loads)
print(d.n_facilities.median(), d.groupby('price_tier').n_facilities.median())
r = d[['price_tier']].join(fac.explode().rename('f')).groupby('f').price_tier.agg(['mean', 'size'])
print(r[r['size'] > 5000].sort_values('mean'))
print(pd.to_datetime(tr.snapshot_start_date).agg(['min', 'max']))
print(d.groupby('price_tier').days_since_last_review.median())
fig, axs = plt.subplots(1, 3, figsize=(12, 3))
for a, c in zip(axs, ['access_type', 'compliance_status', 'operator_class']):
    ct = pd.crosstab(tr[c], tr.price_tier, normalize='index'); print(ct.round(3))
    ct.plot.barh(stacked=True, ax=a, legend=False); a.set_title(c)
plt.tight_layout(); plt.show()

## Q8

In [ ]:
from sklearn.feature_selection import mutual_info_classif
X = d.drop(columns=['price_tier', 'space_id', 'facilities', 'first_review_date', 'last_review_date',
                    'snapshot_end_date', 'operator_ref'])
cat = list(X.select_dtypes(exclude='number').columns)
for c in cat: X[c] = X[c].astype('category').cat.codes
s = X.sample(60000, random_state=0); ys = d.price_tier.loc[s.index]
mi = mutual_info_classif(s.fillna(-999), ys, discrete_features=[c in cat for c in s.columns], random_state=0)
mi = pd.Series(mi, s.columns).sort_values(ascending=False); print(mi.head(20).round(4))
num = d.select_dtypes('number').drop(columns=['space_id'])
print(num.corr('spearman')['price_tier'].drop('price_tier').sort_values(key=abs, ascending=False).head(15).round(3))
print(pd.crosstab(d.max_seats.clip(upper=8), d.price_tier, normalize='index').round(3))
print(d.groupby('price_tier')[['max_seats', 'zones', 'workstations', 'open_slots_c',
                               'booked_load', 'score_overall']].median())
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
mi.head(12)[::-1].plot.barh(ax=axs[0], title='Mutual information')
m9 = d[d.metro == 'MTR-09'].sample(8000, random_state=0)
sc = axs[1].scatter(m9.geo_lon, m9.geo_lat, c=m9.price_tier, s=2); plt.colorbar(sc, ax=axs[1])
axs[1].set_title('MTR-09: location vs tier'); plt.tight_layout(); plt.show()

## Q9

In [ ]:
from scipy.stats import chi2_contingency
c = num.drop(columns=['price_tier']).corr('spearman').abs()
pairs = c.where(np.triu(np.ones(c.shape), 1).astype(bool)).stack().sort_values(ascending=False)
print(pairs[pairs > 0.8].round(3))
sel = ['max_seats', 'workstations', 'zones', 'utility_rooms', 'n_facilities', 'min_term_days', 'min_term_floor',
       'min_term_avg', 'max_term_days', 'max_term_ceiling', 'open_slots_a', 'open_slots_b', 'open_slots_c',
       'open_slots_e', 'review_count', 'review_count_12m', 'review_count_prev_yr', 'booked_load',
       'reviews_per_month', 'score_overall', 'score_value', 'operator_portfolio_size', 'operator_metro_units']
plt.figure(figsize=(9, 7)); sns.heatmap(d[sel].corr('spearman'), cmap='coolwarm', center=0); plt.show()
def cramers_v(a, b):
    t = pd.crosstab(a, b); chi2 = chi2_contingency(t)[0]
    return np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1)))
for a, b in [('space_class', 'access_type'), ('compliance_status', 'metro'), ('access_type', 'metro')]:
    print(a, b, round(cramers_v(tr[a].astype(str), tr[b].astype(str)), 3))
print(tr.groupby('zone').metro.nunique().max(), tr.groupby('borough').metro.nunique().max())
four = d[d.max_seats == 4]
print(pd.crosstab(four.metro, four.price_tier, normalize='index').loc[['MTR-07', 'MTR-09']].round(2))
print(d.groupby('access_type').max_seats.median())
print(d[['n_facilities', 'review_count']].corr('spearman').iloc[0, 1])

## Q10

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
cats = ['metro', 'borough', 'space_class', 'access_type', 'operator_class', 'compliance_status']
X = d.drop(columns=['price_tier', 'space_id', 'facilities', 'first_review_date', 'last_review_date',
                    'snapshot_start_date', 'snapshot_end_date', 'operator_ref'])
X['zone'] = X.zone.astype('category').cat.codes
for c in cats:
    vc = X[c].value_counts()
    X[c] = X[c].where(X[c].map(vc) >= 30, 'rare').astype('category').cat.codes.replace(-1, np.nan)
Xtr, Xva, ytr, yva = train_test_split(X, d.price_tier, test_size=.2, stratify=d.price_tier, random_state=0)
hgb = HistGradientBoostingClassifier(max_iter=200, early_stopping=False, random_state=0,
          categorical_features=[X.columns.get_loc(c) for c in cats]).fit(Xtr, ytr)
p = hgb.predict(Xva)
print('macro F1:', round(f1_score(yva, p, average='macro'), 4), f1_score(yva, p, average=None).round(3))
cm = confusion_matrix(yva, p); print((cm / cm.sum(1, keepdims=True)).round(3))
err = cm.sum() - np.trace(cm)
print('share of errors in adjacent tiers:', sum(cm[i, i+1] + cm[i+1, i] for i in range(3)) / err)